In [7]:
import os
import pandas as pd
import random
import textstat
import re

In [8]:
# Попытка импортировать ruts.readability — если нет, используем свои функции
try:
    from ruts.readability import LIX, RIX, DUB, smog_index as SMOG_ru
except ImportError:
    # Фолбэк-реализации
    def LIX(text):
        sents = [s for s in re.split(r'[.!?]', text) if s.strip()]
        words = re.findall(r'\w+', text)
        if not sents or not words:
            return 0.0
        long_words = [w for w in words if len(w) > 6]
        return len(words) / len(sents) + (len(long_words) * 100.0 / len(words))

    def RIX(text):
        sents = [s for s in re.split(r'[.!?]', text) if s.strip()]
        words = re.findall(r'\w+', text)
        if not sents:
            return 0.0
        long_words = [w for w in words if len(w) > 6]
        return len(long_words) / len(sents)

    def DUB(text):
        vowels = 'аеёиоуыэюя'
        def syllables(w):
            return len(re.findall(rf'[{vowels}]+', w.lower()))
        words = re.findall(r'\w+', text)
        if not words:
            return 0.0
        difficult = [w for w in words if syllables(w) > 2]
        return len(difficult) * 100.0 / len(words)

    def SMOG_ru(text):
        # Для русского текста используем англ. SMOG как приближение
        return textstat.smog_index(text)

In [17]:
# 1. Загрузка текстов за 2023 и 2024 годы
def load_texts_years(base_folder, years=('2023','2024')):
    records = []
    for year in years:
        dir_path = os.path.join(base_folder, f'IMS{year}')
        if not os.path.isdir(dir_path):
            continue
        for fname in os.listdir(dir_path):
            if fname.endswith(f'_IMS_{year}_rus.txt') or fname.endswith(f'_IMS_{year}.txt'):
                path = os.path.join(dir_path, fname)
                with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                    txt = f.read().strip()
                name = os.path.splitext(fname)[0]
                records.append({'Name': name, 'Year': year, 'Text': txt})
    return pd.DataFrame(records)


In [18]:
if __name__ == '__main__':
    base_folder = '/Users/juliak/Downloads/IMS2013-20242'  # измените на ваш путь
    df = load_texts_years(base_folder)

    # Выбираем случайные 10 текстов
    sample = df.sample(n=min(10, len(df)), random_state=42).reset_index(drop=True)

    # Считаем метрики
    results = []
    for _, row in sample.iterrows():
        t = row['Text']
        res = {
            'Name': row['Name'],
            'Year': row['Year'],
            # textstat (англ. метрики)
            'Flesch Reading Ease':        textstat.flesch_reading_ease(t),
            'Flesch-Kincaid Grade':       textstat.flesch_kincaid_grade(t),
            'SMOG (en)':                  textstat.smog_index(t),
            'Coleman-Liau Index':         textstat.coleman_liau_index(t),
            'Automated Readability Index':textstat.automated_readability_index(t),
            'Dale-Chall Score':           textstat.dale_chall_readability_score(t),
            'Gunning Fog':                textstat.gunning_fog(t),
            # Русские метрики (ruts или фолбэк)
            'LIX (ru)':                   LIX(t),
            'RIX (ru)':                   RIX(t),
            'DUB (ru)':                   DUB(t),
            'SMOG (ru)':                  SMOG_ru(t),
        }
        results.append(res)

In [19]:
metrics_df = pd.DataFrame(results)
print("\n=== Readability Metrics Sample ===\n")
print(metrics_df)


=== Readability Metrics Sample ===

                              Name  Year  Flesch Reading Ease  \
0  MitrofanovaGolubev_IMS_2024_rus  2024            99.964223   
1              Sukhan_IMS_2024_rus  2024           103.194765   
2            Khodorkovsky_IMS_2023  2023           103.275802   
3           Vybornaya_IMS_2024_rus  2024           105.219193   
4           ChizhikEgorov_IMS_2023  2023           100.606352   
5              Gavrilkina_IMS_2023  2023            95.638210   
6                Maksimov_IMS_2023  2023           101.848494   
7               Melnichuk_IMS_2023  2023            89.683804   
8              Belkin_IMS_2024_rus  2024           105.348833   
9                 Chizhik_IMS_2023  2023            96.537717   

   Flesch-Kincaid Grade  SMOG (en)  Coleman-Liau Index  \
0              4.026105   5.502600           23.171303   
1              3.006552   4.927825           22.059736   
2              3.494815   3.129100           22.319586   
3              